In [1]:
#Import pandas
import pandas as pd

In [2]:
#Read the CSV from the data folder
df = pd.read_csv("data/raw/hospitalization_discharge.csv")



1. Renamed inpatient_number to patient_id.

Reasoning:
inpatient_number is essentially the unique identifier for each hospitalization/patient record. Renaming it to PatientID makes the dataset easier to understand and use in analysis, dashboards, and joins with other datasets.

In [3]:
df.rename(columns={'inpatient_number': 'patient_id'}, inplace=True)

2. Renamed destinationdischarge to destination_discharge.

Reasoning:
The column describes where the patient went after discharge, such as Home, Healthcare Facility, or Died. Using a clearer name with an underscore improves readability and follows a consistent naming convention.

In [4]:
df.rename(columns={'destinationdischarge': 'destination_discharge'}, inplace=True)

3. Handled the blank values in respiratory_support.

Reasoning:
There are 1,966 blank values in this column. The non-blank values are IMV and NIMV.

Because a blank does not necessarily mean that respiratory information is missing—it can reasonably indicate that the patient did not receive IMV or NIMV—I recommend replacing the blanks with None rather than dropping the records.

This also makes the data easier to analyze because you can distinguish:

IMV → Invasive Mechanical Ventilation
NIMV → Non-Invasive Mechanical Ventilation
None → No IMV/NIMV recorded
Why I prefer None instead of deleting the rows:
There are 1,966 affected records, so dropping them would remove almost the entire dataset for this field and could introduce significant bias.

In [5]:
df['respiratory_support'] = df['respiratory_support'].fillna('None')

4. Renamed dischargeday to discharge_day.

Reasoning:
The existing column name is difficult to read because the words are combined. discharge_day clearly indicates that the value represents the number of days until discharge.

In [6]:
df.rename(columns={'dischargeday': 'discharge_day'}, inplace=True)

5. Verified and standardized admission_date.

Reasoning:
The admission_date values are currently stored as text/object values. All 2,008 records can be successfully interpreted as dates, so there are no invalid date values in this column.

Converting it to a proper datetime format will allow you to easily perform:

Year/month analysis
Admission trends
Time-based filtering
Sorting
Date calculations

In [7]:
df['admission_date'] = pd.to_datetime(
    df['admission_date'],
    errors='coerce'
)


6. Handled the blank values in time_of_death__days_from_admission.

Reasoning:
There are 1,964 blank values in this column. This field represents the number of days from admission until death.

A blank value is meaningful here because most patients did not have a recorded death, so we should not replace the blanks with 0. Replacing blanks with 0 would incorrectly suggest that the patient died on the day of admission.
Created a separate column for easier analysis death_recorded with 1 being Yes and 0 being No

In [8]:
# Keep blanks as NaN because they indicate no recorded time of death
df['time_of_death__days_from_admission'] = pd.to_numeric(
    df['time_of_death__days_from_admission'],
    errors='coerce'
)
df['death_recorded'] = df['time_of_death__days_from_admission'].notna().astype(int)

7. Handled the 1 blank in return_to_emergency_department_within_6_months.

Reasoning:
This is a binary field containing 0 and 1, with only one missing value.

For the one missing record, there isn't enough information in the related fields to confidently determine whether the patient returned to the emergency department. Therefore, I would not automatically convert it to 0, because that would make an assumption about the patient's outcome.

For a clean analytical dataset, we can label the missing value as Unknown.
This results in:

0 → No return to emergency department
1 → Returned to emergency department
Unknown → Information not available

This is safer than filling the blank with 0, because 0 represents an actual outcome, whereas the blank represents missing information.

In [9]:
df['return_to_emergency_department_within_6_months'] = (
    df['return_to_emergency_department_within_6_months']
    .fillna('Unknown')
)

9. Standardized the values in admission_ward from GeneralWard to General Ward.

Reasoning:
Adding the space makes the value more readable and consistent with normal naming conventions. This also prevents GeneralWard and General Ward from being treated as two different categories in Tableau or other analysis.

In [11]:
df['admission_ward'] = df['admission_ward'].replace(
    'GeneralWard',
    'General Ward'
)

10. Standardized the values in admission_way from NonEmergency to Non Emergency.

Reasoning:
Adding the space improves readability and ensures that the same category is consistently represented when creating charts, filters, or counts.

In [12]:
df['admission_way'] = df['admission_way'].replace(
    'NonEmergency',
    'Non Emergency'
)

11. Standardized the values in discharge_department from GeneralWard to General Ward.

Reasoning:
This makes the category easier to read and prevents duplicate categories caused by different formatting of the same value.

In [13]:
df['discharge_department'] = df['discharge_department'].replace(
    'GeneralWard',
    'General Ward'
)

12. Standardized the admission_date format by removing the time portion.

Reasoning:
The current value:

6/16/2016 0:00

contains a time component that is not needed for this analysis. Converting the column to a date-only format gives:

6/16/2016

This makes the field cleaner and easier to use for date-based analysis in Tableau.

In [14]:
df['admission_date'] = pd.to_datetime(
    df['admission_date'],
    errors='coerce'
).dt.strftime('%m/%d/%Y')

13. Standardized numeric columns to integer format while preserving blank values.

Reasoning:
These columns contain whole-number values, but pandas converted them to floating-point format (1.0, 2.0) because of missing values. Converting them to Int64 keeps the values as integers while still allowing blanks, making the data cleaner and more consistent for analysis.

In [16]:
df['readmission_time_days_from_admission'] = pd.to_numeric(
    df['readmission_time_days_from_admission'],
    errors='coerce'
).astype('Int64')

df['return_to_emergency_department_within_6_months'] = pd.to_numeric(
    df['return_to_emergency_department_within_6_months'],
    errors='coerce'
).astype('Int64')

df['time_to_emergency_department_within_6_months'] = pd.to_numeric(
    df['time_to_emergency_department_within_6_months'],
    errors='coerce'
).astype('Int64')

In [17]:
print(df[
    [
        'readmission_time_days_from_admission',
        'return_to_emergency_department_within_6_months',
        'time_to_emergency_department_within_6_months'
    ]
].dtypes)

readmission_time_days_from_admission              Int64
return_to_emergency_department_within_6_months    Int64
time_to_emergency_department_within_6_months      Int64
dtype: object


14. Standardized the oxygen_inhalation values from OxygenTherapy to Oxygen Therapy.

Reasoning:
Adding the space improves readability and ensures the same category is consistently represented in analysis and dashboard visualizations.

In [20]:
df['oxygen_inhalation'] = df['oxygen_inhalation'].replace(
    'OxygenTherapy',
    'Oxygen Therapy'
)

14. Standardized the oxygen_inhalation values from OxygenTherapy to Oxygen Therapy.

Reasoning:
Adding the space improves readability and ensures the same category is consistently represented in analysis and dashboard visualizations.

In [22]:
df['oxygen_inhalation'] = df['oxygen_inhalation'].replace(
    'OxygenTherapy',
    'Oxygen Therapy'
)

15. Standardized categorical values for consistency and readability.

Reasoning:
Added spaces to combined words so the values are easier to read and are treated consistently as single categories in analysis and dashboards.

In [23]:
df['destination_discharge'] = df['destination_discharge'].replace(
    'HealthcareFacility', 'Health care Facility'
)

df['oxygen_inhalation'] = df['oxygen_inhalation'].replace(
    'AmbientAir', 'Ambient Air'
)

df['outcome_during_hospitalization'] = df['outcome_during_hospitalization'].replace(
    'DischargeAgainstOrder', 'Discharge Against Order'
)

In [24]:
# save the cleaned file
df.to_csv(
    "data/cleaned/hospitalization_discharge_cleaned.csv",
    index=False
)
import os

print(os.listdir("data/cleaned"))
cleaned_df = pd.read_csv(
    "data/cleaned/hospitalization_discharge_cleaned.csv"
)

cleaned_df.head()

['hospitalization_discharge_cleaned.csv']


,patient_id,destination_discharge,admission_ward,admission_way,discharge_department,visit_times,respiratory_support,oxygen_inhalation,discharge_day,admission_date,...,re_admission_within_28_days,death_within_3_months,re_admission_within_3_months,death_within_6_months,re_admission_within_6_months,time_of_death__days_from_admission,readmission_time_days_from_admission,return_to_emergency_department_within_6_months,time_to_emergency_department_within_6_months,death_recorded
0,857781,Home,Cardiology,Non Emergency,Cardiology,1,NaN,Oxygen Therapy,11,01/24/2017,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
1,743087,Home,Cardiology,Non Emergency,Cardiology,1,NaN,Oxygen Therapy,8,05/05/2017,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
2,866418,Home,Cardiology,Non Emergency,Cardiology,2,NaN,Oxygen Therapy,5,11/18/2016,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
3,775928,Home,Cardiology,Emergency,Cardiology,1,NaN,Oxygen Therapy,11,10/02/2017,...,1,0,1,0,1,NaN,19.0,1.0,19.0,0
4,810128,Home,Cardiology,Non Emergency,Cardiology,1,NaN,Oxygen Therapy,5,11/17/2019,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
